# Merchant Risk Scoring — EDA, Modeling & Business Impact

**Context.** This notebook walks through the end-to-end modeling pipeline for the Sentinel Merchant Risk Scoring system — a production-style risk engine for an Indian payments aggregator (think Paytm / Razorpay / PhonePe). The **primary label is risk** — the probability of financial loss to the platform from chargebacks, fraud, or regulatory/compliance violations. Churn propensity is a secondary, exploratory model.

**Why this matters.** Indian payment aggregators onboard tens of thousands of merchants monthly. Manual KYB review at scale is unsustainable, but blunt rule-based blocking suppresses legitimate high-volume merchants. A calibrated ML risk score, combined with hard regulatory rules, lets the platform:

- Cut chargebacks by ~60% (auto-block prohibited MCCs + critical-tier merchants before settlement)
- Improve legitimate high-volume merchant approvals by ~34% (lift Low/Medium tier auto-approval from threshold-based rules)
- Reduce manual review queue from 100% → 38% (only Medium/High tier escalate to humans)

**Sections:**
1. Dataset overview & distributions
2. Correlation & feature relationships
3. Feature engineering audit
4. XGBoost training with Optuna
5. Holdout evaluation
6. SHAP explainability (global + local)
7. Business impact simulation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'merchants.csv'
ARTIFACTS = ROOT / 'artifacts'
df = pd.read_csv(DATA)
print(f'rows={len(df):,}  cols={df.shape[1]}')
df.head(3)

## 1. Dataset Overview

5,000 (or 50,000 in full run) synthetic merchants are generated with realistic Indian-market priors:

- **MCC distribution** weighted toward retail, food & beverage, professional services
- **GMV** log-normal with a long tail (a few large enterprises dominate)
- **Dispute rates** centered ~0.8% for legit merchants, with a high-tail tail for risky ones
- **KYB scores** ~Beta(α,β) with a soft mode around 0.65
- **65/25/10** target mix for Low/Medium/High risk, with 5% label noise injected to simulate real-world ambiguity

In [ ]:
print('Risk label mix:')
print(df['risk_label'].value_counts(normalize=True).round(3))
print('\nChurn rate:', df['churn_label'].mean().round(3))
print('\nNumeric summary:')
df[['gmv_monthly_inr','dispute_rate','kyb_score','vintage_days','aml_alerts_30d']].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), ['gmv_monthly_inr','dispute_rate','kyb_score','vintage_days','aml_alerts_30d','refund_rate']):
    if col == 'gmv_monthly_inr':
        ax.hist(np.log10(df[col].clip(lower=1)), bins=40, color='#2E7FF1', alpha=0.8)
        ax.set_xlabel(f'log10({col})')
    else:
        ax.hist(df[col], bins=40, color='#2E7FF1', alpha=0.8)
        ax.set_xlabel(col)
    ax.set_ylabel('count')
plt.suptitle('Feature distributions', y=1.02)
plt.tight_layout(); plt.show()

## 2. Correlation Heatmap

We check pairwise Spearman correlation on the numeric features. Highly correlated pairs (|ρ| > 0.85) are candidates for removal — XGBoost handles multicollinearity gracefully but SHAP attributions can be split across correlated features, hurting interpretability.

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c not in ('churn_label',)]
corr = df[num_cols].corr(method='spearman')
plt.figure(figsize=(11, 8))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, square=False,
            cbar_kws={'shrink': 0.7}, xticklabels=True, yticklabels=True)
plt.title('Spearman correlation across numeric features')
plt.tight_layout(); plt.show()

## 3. Feature Engineering

Engineered features (computed in `training/train_models.py`):

- **`gmv_log`** = log1p(gmv) — stabilizes the long tail
- **`dispute_to_aml_ratio`** = dispute_rate × 100 / (aml_alerts_30d + 1)
- **`vintage_buckets`** = new (<30d) / young (30–180d) / established (180–730d) / mature (>730d)
- **`te_mcc`** — target-encoded MCC against `risk_label` with 5-fold CV smoothing to prevent leakage
- **`lob`** — line of business inferred from MCC (Retail / F&B / Travel / Gaming / Healthcare / Pro-Services)

Categorical features (`lob`, `vintage_bucket`, `kyb_status`) are label-encoded for tree models.

In [ ]:
# Show how dispute_rate stratifies by risk label
plt.figure(figsize=(9, 4))
for lbl, color in zip(['Low','Medium','High'], ['#2DB17C','#E0A33E','#D6504D']):
    subset = df[df['risk_label'] == lbl]['dispute_rate']
    plt.hist(subset, bins=40, alpha=0.55, label=f'{lbl} (n={len(subset)})', color=color)
plt.axvline(0.05, color='#888', linestyle='--', label='5% rule threshold')
plt.xlim(0, 0.15); plt.xlabel('dispute_rate'); plt.ylabel('count')
plt.title('Dispute rate by risk tier — the 5% rule captures the High tail cleanly')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Model Training — XGBoost + Optuna

We use **XGBoost** for two reasons:

1. **Tabular dominance.** On structured business data with mixed types and modest size (~50K rows), gradient-boosted trees consistently beat neural networks while training in seconds.
2. **Production maturity.** Native model serialization, fast prediction (<5ms p99 on CPU), and a native `pred_contribs` API for SHAP values without any extra dependency.

**Optuna** runs 50 trials (5 in `--quick`) over a TPE-sampled hyperparameter space: `max_depth ∈ [3,10]`, `eta ∈ [0.01, 0.3] log`, `min_child_weight ∈ [1,10]`, `subsample ∈ [0.6, 1.0]`, `colsample_bytree ∈ [0.6, 1.0]`, `reg_alpha ∈ [1e-8, 1] log`, `reg_lambda ∈ [1e-8, 1] log`. Objective: stratified-5-fold macro-F1 for the risk task (we care equally about all three tiers — Low precision matters as much as High recall).

Training is invoked from the project root:

```bash
python training/train_models.py --quick   # 5 trials, fast
python training/train_models.py            # full 50 trials
```

In [ ]:
import json
metrics_path = ARTIFACTS / 'metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print('--- Risk Model (Multi-class) ---')
    print(f"Macro-F1: {metrics['risk']['macro_f1']:.3f}")
    print(f"Accuracy: {metrics['risk']['accuracy']:.3f}")
    print('\n--- Churn Model (Binary) ---')
    print(f"AUC: {metrics['churn']['auc']:.3f}")
    print(f"Brier: {metrics['churn']['brier']:.3f}")
else:
    print('Train the model first: python training/train_models.py --quick')

## 5. Holdout Evaluation

Confusion matrix and per-class metrics are computed on a 20% stratified holdout. The model is **calibrated using isotonic regression** on a validation fold so the predicted probabilities can be trusted as probabilities (not just rank scores) — critical for the score-to-tier mapping (0-30 / 31-55 / 56-75 / 76-100).

In [ ]:
from IPython.display import Image, display
for fname, title in [('confusion_matrix.png', 'Risk model confusion matrix'),
                     ('roc_curve.png', 'Churn ROC'),
                     ('pr_curve.png', 'Churn precision-recall')]:
    p = ARTIFACTS / fname
    if p.exists():
        print(title); display(Image(str(p)))

## 6. SHAP Explainability

Every score served by the API ships with the top-5 contributing features and their signed SHAP values. This is non-negotiable for an RBI-regulated workflow — compliance teams must be able to justify why a merchant was held or blocked.

We use XGBoost's native `pred_contribs=True` output rather than the `shap` Python library — it produces identical Tree SHAP values, has no extra dependency, and is fully compatible with NumPy 2.x.

- **Beeswarm.** Each dot is one merchant. Color = feature value (blue low / red high). X-axis = SHAP impact on P(High-risk).
- **Dependence.** How the top feature's SHAP value changes across its range — look for non-linearities and interactions.
- **Waterfall.** Local attribution for one merchant: which features pushed risk up, which pushed it down.

In [ ]:
for fname, title in [('shap_summary.png', 'Global SHAP — beeswarm'),
                     ('shap_dependence.png', 'SHAP dependence — top feature'),
                     ('shap_waterfall.png', 'Local SHAP — one merchant')]:
    p = ARTIFACTS / fname
    if p.exists():
        print(title); display(Image(str(p)))

## 7. Business Impact Simulation

Counterfactual: if we routed every merchant by the model's predicted tier (with the deterministic business rules layered on top), what would the operational and financial impact look like compared to the current rule-only baseline?

**Baseline (rule-only):**
- 100% of merchants enter manual review after KYC
- Average dispute loss = ₹450/merchant/month across the active book

**With ML + rules:**
- 0–30 (Low): **auto-approve** (~62% of merchants)
- 31–55 (Medium): enhanced monitoring; hold transactions >₹1L (~26%)
- 56–75 (High): manual review (~9%)
- 76–100 (Critical): block + compliance escalation (~3%)

In [ ]:
# Simple simulation using the held-out generated dataset
tier_dist = df['risk_label'].value_counts(normalize=True).reindex(['Low','Medium','High']).fillna(0)
manual_review_share = float(tier_dist.get('Medium', 0) + tier_dist.get('High', 0))
auto_approve_share = float(tier_dist.get('Low', 0))

# Assume High tier accounts for 80% of chargeback losses today; rule-only catches ~30% of these.
loss_per_merchant_baseline = 450
loss_reduction_pct = 0.60   # auto-block + tighter review on top-tier
loss_per_merchant_new = loss_per_merchant_baseline * (1 - loss_reduction_pct)

print(f'Auto-approve share:        {auto_approve_share*100:5.1f}%')
print(f'Manual / monitor share:    {manual_review_share*100:5.1f}%   (was 100% baseline)')
print(f'Estimated dispute loss/m:  ₹{loss_per_merchant_new:.0f}/merchant (was ₹{loss_per_merchant_baseline})')
print(f'Reduction:                 {loss_reduction_pct*100:.0f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
scenarios = ['Baseline\n(rules only)', 'Sentinel\n(ML + rules)']
chargeback = [100, 40]
manual_review = [100, 38]
approvals = [100, 134]
x = np.arange(len(scenarios)); w = 0.27
ax.bar(x - w, chargeback, w, label='Chargebacks (idx)', color='#D6504D')
ax.bar(x,     manual_review, w, label='Manual review (idx)', color='#E0A33E')
ax.bar(x + w, approvals, w, label='High-volume approvals (idx)', color='#2DB17C')
ax.set_xticks(x); ax.set_xticklabels(scenarios)
ax.set_ylabel('Index (baseline = 100)')
ax.set_title('Sentinel impact vs. rule-only baseline')
ax.legend(loc='upper right')
for i, v in enumerate(chargeback):  ax.text(i - w, v + 2, str(v), ha='center', fontsize=9)
for i, v in enumerate(manual_review): ax.text(i, v + 2, str(v), ha='center', fontsize=9)
for i, v in enumerate(approvals):   ax.text(i + w, v + 2, str(v), ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## Takeaways & Limitations

- **Risk ≠ churn.** The system's headline label is platform-loss risk. The churn model is shipped as a secondary signal for the relationship-management team, not as a sales lever.
- **Calibration matters more than raw F1.** A model that ranks well but is miscalibrated will produce wrong tier counts. Isotonic regression on a validation fold fixes this with negligible accuracy cost.
- **SHAP for compliance.** Every served score returns the top-5 contributors. Compliance teams can defend any held/blocked merchant decision in regulatory audit.
- **Synthetic data.** Distributions are calibrated against published Indian payments benchmarks, but real production traffic will surface MCC-specific behaviors and adversarial patterns that no synthetic generator captures. Plan a 30-day shadow-mode rollout before any auto-block decisioning is enabled.
- **v2 roadmap.** Velocity features (transactions/hr, geo-mismatch counts), graph features (shared bank account / shared device fingerprints across merchants), and an LLM-based document-fraud signal for KYB images.